In [ ]:
%%html
<style>
    h1 {color:purple}
    h2 {color:purple}
    h3 {color:#0099ff}
    hr {
        border: 0;
        height: 3px;
        background: #333;
        background-image: linear-gradient(
            to right,
            limegreen,
            deepskyblue,
            limegreen
        );
    }
</style>

# Creating Agents with the OpenAI Agents SDK
---
# `ShellTool` Folder Inspector
* `ShellTool` gives agents access to shell commands through an executor you provide
* Commands can "touch" the local machine
* Useful for local project inspection, file discovery/manipulation, ...
    * We'll use it to inspect a local folder containing Python files
* This demo allows only simple read-only commands
* Agent summarizes Python files, their likely dependencies and run commands in a Markdown document
    * No files written in this demo
    * Use human-in-the-loop for commands that can make changes
* Production code using this tool should have strict path controls, command review and human approval

---


## Demo Workflow

* Pick a small local folder containing Python examples
* Let the agent run read-only shell commands such as `pwd`, `ls`, `find` and `sed -n`
* Have the agent infer likely third-party dependencies from imports
* Display a short Markdown summary in the notebook

---


## Imports and Constants

In [ ]:
from pathlib import Path

from IPython.display import Markdown, display
from agents import (
    Agent,        # defines the folder-inspection agent
    ModelSettings,
    Runner,
    ShellTool,    # exposes shell commands as an agent tool
    trace
)

# Edit this path before running the demo
TARGET_CODE_FOLDER = Path('resources') / 'python_examples'

---

## `ReadOnlyShellExecutor`

A safe, allowlist-based executor that sits between the agent and the OS.

* Accepts only a short list of read-only command prefixes (`ls`, `cat`, `find`, etc.)
* Rejects any command containing shell-injection characters (`|`, `>`, `;`, `&&`, ...)
* Blocked commands return an error message to the agent instead of running
* Full source: [Open shell_executor.py](./source/shell_executor.py)

---

In [ ]:
from source.shell_executor import ReadOnlyShellExecutor

# set needs_approval to True for human-in-the-loop
shell = ShellTool(executor=ReadOnlyShellExecutor(), needs_approval=False)

---

## The Agent

* Prompt tells the agent which read-only commands it may use
* The final answer should be useful Markdown, not raw command output
* The agent should not run examples or write files


In [ ]:
folder_inspector_agent = Agent(
    name='ShellTool Folder Inspector',
    model='gpt-5.6-terra',
    model_settings=ModelSettings(
        reasoning={'effort': 'low'}, 
        verbosity='low'
    ),
    tools=[shell],
    instructions="""You inspect a small local Python example folder using
        the ShellTool only for read-only inspection. Do not run Python 
        scripts. Do not write, edit, delete, move or copy files. 
        Run ONE simple command per shell call: no pipes, no ; or &&
        chaining, no redirection. Allowed commands: pwd, ls, find,
        head, cat, sed -n, grep, wc. Prefer grep to find imports.
        Return concise Markdown with: likely third-party dependencies, 
        suggested run commands and any important assumptions."""
)

---

## Run the Demo

* The trace name identifies this notebook in the traces dashboard
* If the agent asks for a blocked command, the executor returns an error and the agent must choose a safer command
* The displayed result is the teaching artifact for the class


In [ ]:
prompt = f"""
Inspect this local folder:
{TARGET_CODE_FOLDER}

Use read-only shell commands only. Produce a concise Markdown summary with:
- Python files discovered
- Third-party dependencies inferred from imports
- Suggested commands for running the examples
- Any assumptions or limitations
"""

with trace('02-05-08-shell-tool-folder-inspector'):
    # each shell call is one turn; give the inspector room for a few files
    result = await Runner.run(folder_inspector_agent, prompt, max_turns=30)

display(Markdown(result.final_output))

---

## References

* [OpenAI Agents SDK: Tools](https://openai.github.io/openai-agents-python/tools/)
* [OpenAI Agents SDK: Human in the loop](https://openai.github.io/openai-agents-python/human_in_the_loop/)
* [SDK example: shell_human_in_the_loop.py](https://github.com/openai/openai-agents-python/blob/main/examples/tools/shell_human_in_the_loop.py)

---


© 2026 by Deitel & Associates, Inc. All Rights Reserved.